## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, trace, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from agents.model_settings import ModelSettings
from openai import AsyncOpenAI
from ddgs import DDGS
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import os
from IPython.display import display, Markdown
from messenger import send_email, push

In [3]:
load_dotenv(override=True)

True

In [30]:
# Constants 

MODEL_NAME = "mimo-v2.6:9b"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

# Run every agent on local Ollama via its OpenAI-compatible endpoint
set_tracing_disabled(True)  # no OpenAI key, so don't upload traces to platform.openai.com
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")  # api_key is required by the client, ignored by Ollama
ollama_model = OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=ollama_client)
email_model = OpenAIChatCompletionsModel(model="qwen3:8b", openai_client=ollama_client)

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing

### Local alternative used in this notebook

Hosted tools only work with OpenAI models through the Responses API. Since the agents here run on local Ollama, we swap `WebSearchTool` for our own `web_search` function tool, which queries DuckDuckGo with the [`ddgs`](https://github.com/deedy5/ddgs) library. It's free and needs no API key.

In [5]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you use the web_search tool to search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"


@function_tool
async def web_search(query: str) -> str:
    """
    Search the web and return the top results (title, URL and snippet for each).

    Args:
        query: The search term
    """
    def _search():
        return DDGS().text(query, max_results=8)
    try:
        # ddgs is synchronous - run it in a thread so parallel searches don't block each other
        results = await asyncio.to_thread(_search)
    except Exception as e:
        return f"Search failed: {e}"
    if not results:
        return "No results found."
    return "\n\n".join(f"{r['title']}\n{r['href']}\n{r['body']}" for r in results)


settings = ModelSettings(tool_choice="required")
tools = [web_search]

In [6]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=ollama_model, model_settings=settings)

In [7]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

As of 2026, the dominant AI agent frameworks cluster around three ecosystems: **LangGraph** (LangChain) is generally considered the most mature choice for stateful, long-running, multi-step agents, with strong tracing via LangSmith and support in both Python and TypeScript. **CrewAI** remains the go-to for role-based multi-agent "crews" and quick prototypes, while **Microsoft AutoGen** (now AG2) is favored for enterprise/enterprise-co-pilot work, especially with human-in-the-loop and skills-based orchestration. **OpenAI Agents SDK** and the **OpenAI Assistants API** lead the lightweight, app-level agent space (tool use, real-time agents), and **Anthropic's Claude Agent SDK** is the pick for Claude-backed coding agents.

Other notable options include **Mastra** (TypeScript-first, good for full-stack apps), **Google ADK**, **Pydantic AI** (type-safe agents), and **Amazon Bedrock Agents** for AWS-native deployments.

Consensus across 2026 comparisons (LangChain's own review, toolradar, softmaxdata, and various developer guides) is that LangGraph wins on production readiness and observability, CrewAI wins on speed for multi-agent setups, and AutoGen is strongest for Microsoft-centric enterprise workflows. Choosing depends mostly on model/provider preference, state-management needs, and whether you need human-in-the-loop control.

### As always, take a look at the trace

https://platform.openai.com/traces

(Tracing is disabled in this notebook because the agents run on local Ollama, so nothing is uploaded there.)

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [8]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [9]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [10]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=ollama_model, output_type=WebSearchPlan)

In [13]:
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Compare the leading AI agent frameworks (LangGraph, AutoGen, CrewAI, OpenAI Swarm, etc.) and identify which has the most usage and community adoption in 2026.', query='most popular AI agent frameworks 2026 comparison LangGraph AutoGen CrewAI Swarm'), WebSearchItem(reason='Find current usage statistics, GitHub stars, and download numbers for major AI agent frameworks to determine which is most popular.', query='AI agent framework GitHub stars usage statistics 2026 LangGraph AutoGen CrewAI'), WebSearchItem(reason='Find the current state of the OpenAI Swarm agent framework and how it compares to other agent frameworks in 2026.', query='OpenAI Swarm agent framework 2026 features alternatives'), WebSearchItem(reason='Find benchmark results and performance comparisons across AI agent frameworks to determine which are most effective and popular in 2026.', query='AI agent framework benchmarks leaderboard 2026'), WebSearchItem(reason='Find recent ne

## Agent 3: The Writer Agent

In [24]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 1-2 pages of content, around 300-500 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=ollama_model, output_type=ReportData)

## Agent 4: The email agent

In [33]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    print("Calling send email tool")
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [16]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [31]:
INSTRUCTIONS = """
You are provided with a detailed report. Use send_email_tool tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line. You have to use send_email_tool tool, it must be called.
"""

settings = ModelSettings(tool_choice="required")

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=email_model, model_settings=settings)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [20]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [22]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [35]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


### As always, take a look at the trace

https://platform.openai.com/traces

(Tracing is disabled in this notebook because the agents run on local Ollama, so nothing is uploaded there.)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>